# 04 · Morphological & colour feature extraction

Measures per-grain length, width, aspect ratio, area, perimeter, circularity,
solidity, roughness and **colour** (mean CIE-Lab + ΔE) from the segmented images and
writes one master `feature_values.csv`.

### Calibration
The two subsets were imaged in different campaigns and have **different µm/px**. Values are
read from `config.py`:

* **Basmati / varietal subset — `CALIBRATION_BASMATI = 1.79 µm/px`** 
* **BPT 5204 — `CALIBRATION_BPT = 1.09µm/px`**
Raw **pixel** columns (`length_px`, `width_px`, `area_px`) are stored alongside the mm
values plus `calibration_um_per_px`, so if a calibration changes you recompute the mm
columns from the CSV — no re-extraction, and no separate correction script needed.


## Setup

In [2]:
import sys, os
for pth in (os.path.abspath("."), os.path.abspath("..")):
    if pth not in sys.path:
        sys.path.append(pth)

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from skimage.color import rgb2lab

import config as cfg
import rice_common as rc

BPT_SUBDIR     = rc.BPT_SUBDIR                 # "BPT5204"
BASMATI_SUBDIR = "Basmati and non-basmati"

FG_THRESHOLD = 12                              # foreground brightness cutoff (kills JPEG halo)
MIN_AREA     = cfg.MIN_AREA

OUT_DIR = cfg.OUTPUT_DIR / "features"
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = OUT_DIR / "feature_values.csv"

print("BPT     calibration:", cfg.CALIBRATION_BPT, "um/px ->", cfg.CALIBRATION_BPT_mm, "mm/px")
print("Basmati calibration:", cfg.CALIBRATION_BASMATI, "um/px ->", cfg.CALIBRATION_BASMATI_mm, "mm/px")
if abs(cfg.CALIBRATION_BASMATI - 1.79) > 1e-6:
    print("\n[!] CALIBRATION_BASMATI is not 1.79 - update config.py (Section 4 confirms 1.79).")

BPT     calibration: 1.05 um/px -> 0.0010500000000000002 mm/px
Basmati calibration: 1.79 um/px -> 0.0017900000000000001 mm/px


## 1 · Calibration & label resolver

Calibration depends on the **subset**, not the class. Labels come from the folder; the
filename acronym is only cross-checked.

In [2]:
def resolve(path):
    # Return (dataset, variety, class_label, um_per_px, mm_per_px) for an image path.
    parts = Path(path).parts
    if BPT_SUBDIR in parts:
        folder = Path(path).parent.name
        label  = rc.BPT_DIR_TO_LABEL.get(folder, folder)
        return "BPT5204", "BPT5204", label, cfg.CALIBRATION_BPT, cfg.CALIBRATION_BPT_mm
    elif BASMATI_SUBDIR in parts:
        variety = Path(path).parent.name
        return "Basmati", variety, variety, cfg.CALIBRATION_BASMATI, cfg.CALIBRATION_BASMATI_mm
    else:
        raise ValueError(f"cannot resolve subset for: {path}")

def list_segmented():
    # Deterministic list of every segmented grain image (both subsets).
    root = Path(cfg.SEGMENTED_IMAGE_DIR)
    paths = [p for p in root.rglob("*")
             if p.is_file() and p.suffix.lower() in rc.VALID_EXT]
    paths.sort(key=lambda p: str(p))
    return paths

paths = list_segmented()
from collections import Counter
print(f"segmented images: {len(paths)} | by dataset:", dict(Counter(resolve(p)[0] for p in paths)))

segmented images: 24222 | by dataset: {'BPT5204': 14228, 'Basmati': 9994}


## 2 · Feature extraction

Shape metrics from the largest grain contour; colour from grain pixels only. Ratio metrics
(aspect ratio, circularity, solidity, roughness) are unit-free; only length/width/area/
perimeter scale with calibration.

In [3]:
def grain_mask(img_rgb):
    # Robust foreground mask: brightest channel over threshold, largest component, holes filled.
    bright = img_rgb.max(axis=2)
    m = (bright > FG_THRESHOLD).astype(np.uint8) * 255
    n, lab, stats, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return None
    largest = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    m = np.where(lab == largest, 255, 0).astype(np.uint8)
    ff = m.copy(); h, w = m.shape
    cv2.floodFill(ff, np.zeros((h + 2, w + 2), np.uint8), (0, 0), 255)
    return cv2.bitwise_or(m, cv2.bitwise_not(ff))

def extract_features(img_path):
    img = cv2.imread(str(img_path))
    if img is None:
        return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    mask = grain_mask(img_rgb)
    if mask is None or cv2.countNonZero(mask) < MIN_AREA:
        return None
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    area_px = cv2.contourArea(cnt)
    if area_px < MIN_AREA:
        return None
    perim_px = cv2.arcLength(cnt, True)
    (_, _), (w, h), _ = cv2.minAreaRect(cnt)
    length_px, width_px = max(w, h), min(w, h)

    hull = cv2.convexHull(cnt)
    convex_area  = cv2.contourArea(hull)
    convex_perim = cv2.arcLength(hull, True)
    solidity    = area_px / convex_area  if convex_area  > 0 else np.nan
    roughness   = perim_px / convex_perim if convex_perim > 0 else np.nan
    circularity = (4 * np.pi * area_px) / (perim_px ** 2) if perim_px > 0 else np.nan
    aspect_ratio = length_px / width_px if width_px > 0 else np.nan

    grain_px = img_rgb[mask > 0]
    lab = rgb2lab(grain_px.reshape(-1, 1, 3)).reshape(-1, 3)
    mL, ma, mb = lab.mean(axis=0)

    dataset, variety, label, um_px, mm_px = resolve(img_path)
    return {
        "dataset": dataset, "variety": variety, "class": label,
        "image": Path(img_path).name, "grain_id": rc.grain_id(img_path),
        "length_px": length_px, "width_px": width_px, "area_px": area_px,
        "length_mm": length_px * mm_px, "width_mm": width_px * mm_px,
        "area_mm2": area_px * (mm_px ** 2), "perimeter_mm": perim_px * mm_px,
        "aspect_ratio": aspect_ratio, "circularity": circularity,
        "solidity": solidity, "roughness_index": roughness,
        "mean_L": mL, "mean_a": ma, "mean_b": mb,
        "calibration_um_per_px": um_px,
    }

for p in paths:
    demo = extract_features(p)
    if demo:
        print("sample row:", {k: (round(v, 3) if isinstance(v, float) else v)
                               for k, v in list(demo.items())[:12]})
        break

sample row: {'dataset': 'BPT5204', 'variety': 'BPT5204', 'class': 'Broken', 'image': 'BPT5204_BK_0001.JPG', 'grain_id': 'BPT5204_BK_0001', 'length_px': 2714.0, 'width_px': 2006.0, 'area_px': 4577833.5, 'length_mm': 2.85, 'width_mm': 2.106, 'area_mm2': 5.047, 'perimeter_mm': 9.059}


## 3 · Run over all grains → `feature_values.csv`

In [4]:
rows = [f for f in (extract_features(p) for p in tqdm(paths, desc="extracting", unit="img"))
        if f is not None]
df = pd.DataFrame(rows)
print("extracted:", df.shape)

# ΔE within each dataset: BPT referenced to Healthy-class mean (tracks discolouration),
# basmati referenced to its own subset mean.
df["deltaE"] = np.nan
for ds, sub in df.groupby("dataset"):
    if "Healthy" in set(sub["class"]):
        ref = sub.loc[sub["class"] == "Healthy", ["mean_L", "mean_a", "mean_b"]].mean().values
    else:
        ref = sub[["mean_L", "mean_a", "mean_b"]].mean().values
    df.loc[sub.index, "deltaE"] = np.linalg.norm(
        df.loc[sub.index, ["mean_L", "mean_a", "mean_b"]].values - ref, axis=1)

df = df.sort_values(["dataset", "class", "grain_id"]).reset_index(drop=True)
df.to_csv(OUTPUT_CSV, index=False)
print("wrote", OUTPUT_CSV)
df.head()

extracting: 100%|██████████| 24222/24222 [12:00:58<00:00,  1.79s/img]  


extracted: (24222, 20)
wrote Outputs\features\feature_values.csv


,dataset,variety,class,image,grain_id,length_px,width_px,area_px,length_mm,width_mm,...,perimeter_mm,aspect_ratio,circularity,solidity,roughness_index,mean_L,mean_a,mean_b,calibration_um_per_px,deltaE
0,BPT5204,BPT5204,Broken,BPT5204_BK_0001.JPG,BPT5204_BK_0001,2714.0,2006.0,4577833.5,2.84970,2.10630,...,9.058549,1.352941,0.772914,0.993716,1.059122,60.599098,0.159549,0.018367,1.05,1.346926
1,BPT5204,BPT5204,Broken,BPT5204_BK_0002.JPG,BPT5204_BK_0002,2964.0,1695.0,4270417.0,3.11220,1.77975,...,9.058109,1.748673,0.721080,0.995570,1.068362,62.576531,0.073820,-0.430730,1.05,2.825717
2,BPT5204,BPT5204,Broken,BPT5204_BK_0003.JPG,BPT5204_BK_0003,2822.0,1709.0,4092691.5,2.96310,1.79445,...,8.902917,1.651258,0.715374,0.993779,1.082384,60.044766,0.126996,-0.238185,1.05,0.883588
3,BPT5204,BPT5204,Broken,BPT5204_BK_0004.JPG,BPT5204_BK_0004,2652.0,1986.0,4246619.0,2.78460,2.08530,...,9.173999,1.335347,0.699060,0.979695,1.095499,61.088017,-0.000836,-0.432353,1.05,1.406378
4,BPT5204,BPT5204,Broken,BPT5204_BK_0005.JPG,BPT5204_BK_0005,2683.0,1938.0,4155924.5,2.81715,2.03490,...,8.630776,1.384417,0.772959,0.996053,1.079480,59.510999,0.089164,-0.046340,1.05,1.068662


## 4 · Varietal morphology summary (Table 3)

In [3]:
df = pd.read_csv(OUTPUT_CSV)
summary = (df.groupby(["dataset", "variety"])
             .agg(n=("grain_id", "size"),
                  length_mm=("length_mm", "mean"), length_sd=("length_mm", "std"),
                  width_mm=("width_mm", "mean"),   width_sd=("width_mm", "std"),
                  aspect=("aspect_ratio", "mean"), aspect_sd=("aspect_ratio", "std"),
                  area_mm2=("area_mm2", "mean"))
             .round(3).reset_index())
summary.to_csv(OUT_DIR / "morphology_summary.csv", index=False)
print("wrote", OUT_DIR / "morphology_summary.csv")
summary

wrote Outputs\features\morphology_summary.csv


,dataset,variety,n,length_mm,length_sd,width_mm,width_sd,aspect,aspect_sd,area_mm2
0,BPT5204,BPT5204,14228,4.965,0.577,1.758,0.108,2.834,0.351,6.938
1,Basmati,PB1121,4994,8.576,0.575,1.925,0.126,4.471,0.370,13.622
2,Basmati,PB1401,2499,8.446,0.447,1.878,0.109,4.514,0.374,13.195
3,Basmati,PR14,2501,7.168,0.397,2.023,0.114,3.551,0.234,11.864


---
### Outputs
* `Outputs/features/feature_values.csv` — one row per grain